# Notebook 02 — Data Understanding and Data Auditing

**Project**: Music, Brain & Wellbeing: A Data-Driven Study of Human Responses to Music
**Stage**: Data Understanding (before any cleaning or modelling)

The goal of this notebook is to understand the raw datasets from first principles.
We do not modify, clean, or model anything here.
We only observe, measure, and document what exists.


---
# Phase 1 — Locate and Verify Data

Before loading anything into Pandas, we verify what files exist in `data/raw/`.

**Why raw data must remain unchanged:**
Raw data is the only ground truth we have. Every analysis decision is traceable back to it.
If we modify the raw file, we lose the ability to reproduce our results from scratch.
Data scientists always treat raw data as immutable — all transformations happen in code, not in the file.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Show all CSV files in data/raw/
RAW_DIR = "data/raw"

print("Files in data/raw/:")
print("-" * 50)
for filename in os.listdir(RAW_DIR):
    if filename.endswith(".csv"):
        filepath = os.path.join(RAW_DIR, filename)
        size_bytes = os.path.getsize(filepath)
        size_kb = size_bytes / 1024
        print(f"  {filename}")
        print(f"    Size: {size_bytes:,} bytes ({size_kb:.1f} KB)")
print()
print("Total CSV files found:", sum(1 for f in os.listdir(RAW_DIR) if f.endswith(".csv")))


In [ ]:
# Confirm both files load without errors
MXMH_PATH = "data/raw/mxmh_survey_results.csv"
MBW_PATH  = "data/raw/music_brain_wellbeing.csv"

df_mxmh = pd.read_csv(MXMH_PATH)
df_mbw  = pd.read_csv(MBW_PATH)

print("mxmh_survey_results.csv  — loaded successfully:", df_mxmh.shape)
print("music_brain_wellbeing.csv — loaded successfully:", df_mbw.shape)


---
# Phase 2 — Load and Inspect Each Dataset

We inspect each dataset using 9 standard Pandas operations.
For every operation we explain what Pandas is doing, why we are doing it, and what question it answers.


## Dataset 1: mxmh_survey_results.csv

This is the primary dataset. It is a public survey dataset about music listening habits
and self-reported mental health collected from participants on r/SurveyExchange and other communities.


In [ ]:
# 1. shape
# df.shape returns a tuple (n_rows, n_cols).
# Pandas reads the index of rows and the number of column headers to compute this.
# Question answered: How many observations and how many variables do we have?
rows, cols = df_mxmh.shape
print("Shape:", df_mxmh.shape)
print(f"  -> {rows} participants (rows)")
print(f"  -> {cols} variables (columns)")


In [ ]:
# 2. first 5 rows
# df.head() retrieves the first N rows (default 5).
# Question answered: What does a real row of data look like?
print("First 5 rows:")
df_mxmh.head()


In [ ]:
# 3. last 5 rows
# df.tail() retrieves the last N rows (default 5).
# Question answered: Does the dataset terminate correctly, or is the end truncated?
print("Last 5 rows:")
df_mxmh.tail()


In [ ]:
# 4. column names
# df.columns returns an Index object of all column header strings.
# Question answered: What variables exist, and are any names unexpected?
print("Column names:")
for i, col in enumerate(df_mxmh.columns):
    print(f"  [{i:>2}] {col}")


In [ ]:
# 5. data types
# df.dtypes returns the inferred storage type of each column.
#   float64 = decimal numbers
#   int64   = whole numbers
#   object  = text / strings / mixed
# Question answered: Are numeric columns stored as numbers? Are text columns stored as strings?
print("Data types:")
print(df_mxmh.dtypes.to_string())


In [ ]:
# 6. descriptive statistics
# df.describe() computes count, mean, std, min, 25th/50th/75th percentile, max
# for all numeric columns. Pandas ignores NaN values automatically.
# Question answered: What is the typical range and spread of numerical variables?
print("Descriptive statistics (numeric columns):")
print(df_mxmh.describe().round(2).to_string())


In [ ]:
# 7. unique values per column
# df.nunique() counts distinct unique values per column.
# Question answered: Is a column truly categorical (few unique values) or continuous (many)?
# A binary column will show 2; an ID column might show the same number as rows.
print("Unique values per column:")
print(df_mxmh.nunique().to_string())


In [ ]:
# 8. missing values per column
# Step 1: df.isnull() creates a boolean DataFrame.
#         True  = the value is missing (NaN or None)
#         False = the value is present
# Step 2: .sum() adds up the True values down each column (True counts as 1).
# Result: the total count of missing values per column.
# Question answered: Which columns have gaps, and how many?
missing_count = df_mxmh.isnull().sum()
print("Missing value counts:")
print(missing_count.to_string())


In [ ]:
# 9. percentage of missing values
# Divide the missing count by total rows, then multiply by 100.
# This normalizes across datasets of different sizes.
# Question answered: How severe is each gap (e.g. 1% vs 15%)?
missing_pct = (df_mxmh.isnull().sum() / len(df_mxmh) * 100).round(2)
print("Missing value percentage per column:")
print(missing_pct.to_string())


## Dataset 2: music_brain_wellbeing.csv

This is a small synthetic sample dataset created for practice.
It contains 25 simulated participants (plus one accidental duplicate row).
It includes demographic, music behaviour, physiological (EEG, heart rate), and wellbeing variables.


In [ ]:
# 1-9. Applying all inspection operations to the second dataset

print("=== music_brain_wellbeing.csv ===")
print()

print("1. shape:", df_mbw.shape)
print()

print("2. First 5 rows:")
print(df_mbw.head().to_string())
print()

print("3. Last 5 rows:")
print(df_mbw.tail().to_string())
print()

print("4. Column names:")
for i, col in enumerate(df_mbw.columns):
    print(f"  [{i:>2}] {col}")
print()

print("5. Data types:")
print(df_mbw.dtypes.to_string())
print()

print("6. Descriptive statistics:")
print(df_mbw.describe().round(2).to_string())
print()

print("7. Unique values per column:")
print(df_mbw.nunique().to_string())
print()

print("8. Missing value counts:")
print(df_mbw.isnull().sum().to_string())
print()

print("9. Missing value percentage:")
missing_pct_mbw = (df_mbw.isnull().sum() / len(df_mbw) * 100).round(2)
print(missing_pct_mbw.to_string())


---
# Phase 3 — Understand Each Column

We inspect every column of `mxmh_survey_results.csv` individually and classify it.


## Column Classification Key

- **A** — Music behaviour (listening habits, preferences, context)
- **B** — Demographic information (age, gender equivalents)
- **C** — Mental health / wellbeing (self-reported symptoms, scores)
- **D** — Another type of variable
- **E** — Unclear / metadata — Needs verification


In [ ]:
# Inspect each column of df_mxmh
columns_info = []

for col in df_mxmh.columns:
    col_type = str(df_mxmh[col].dtype)
    n_unique = df_mxmh[col].nunique()
    missing = df_mxmh[col].isnull().sum()
    missing_pct = round(missing / len(df_mxmh) * 100, 2)
    examples = df_mxmh[col].dropna().unique()[:4].tolist()
    columns_info.append({
        'Column': col,
        'dtype': col_type,
        'n_unique': n_unique,
        'missing': missing,
        'missing_pct': missing_pct,
        'examples': examples
    })

for c in columns_info:
    print(f"Column  : {c['Column']}")
    print(f"  dtype    : {c['dtype']}")
    print(f"  unique   : {c['n_unique']}")
    print(f"  missing  : {c['missing']} ({c['missing_pct']}%)")
    print(f"  examples : {c['examples']}")
    print()


## Column Interpretations — mxmh_survey_results.csv

| # | Column | Meaning | Type | Category | Missing % | Notes |
|---|---|---|---|---|---|---|
| 0 | `Timestamp` | Survey submission date/time | str | E — Metadata | 0% | Not a feature. Unique-ish. |
| 1 | `Age` | Participant age in years | float64 | B — Demographic | 0.14% | Range 10–89. 1 missing. |
| 2 | `Primary streaming service` | Main music platform used | str | A — Music behaviour | 0.14% | 6 unique values |
| 3 | `Hours per day` | Hours spent listening per day | float64 | A — Music behaviour | 0% | Range 0–24 |
| 4 | `While working` | Listens to music while working (Yes/No) | str | A — Music behaviour | 0.41% | Binary |
| 5 | `Instrumentalist` | Plays an instrument (Yes/No) | str | A — Music behaviour | 0.54% | Binary |
| 6 | `Composer` | Composes music (Yes/No) | str | A — Music behaviour | 0.14% | Binary |
| 7 | `Fav genre` | Favourite music genre | str | A — Music behaviour | 0% | 16 genres |
| 8 | `Exploratory` | Explores new genres actively (Yes/No) | str | A — Music behaviour | 0% | Binary |
| 9 | `Foreign languages` | Listens in foreign languages (Yes/No) | str | A — Music behaviour | 0.54% | Binary |
| 10 | `BPM` | Self-reported favourite tempo (beats per min) | float64 | A — Music behaviour | **14.54%** | Problematic: extreme outliers (999999999) |
| 11–26 | `Frequency [genre]` x 16 | How often each genre is listened to | str | A — Music behaviour | 0% | Ordinal: Never / Rarely / Sometimes / Very frequently |
| 27 | `Anxiety` | Self-reported anxiety score 0–10 | float64 | C — Mental health | 0% | Self-report, NOT clinical diagnosis |
| 28 | `Depression` | Self-reported depression score 0–10 | float64 | C — Mental health | 0% | Self-report, NOT clinical diagnosis |
| 29 | `Insomnia` | Self-reported insomnia score 0–10 | float64 | C — Mental health | 0% | Self-report, NOT clinical diagnosis |
| 30 | `OCD` | Self-reported OCD score 0–10 | float64 | C — Mental health | 0% | Self-report, NOT clinical diagnosis |
| 31 | `Music effects` | Perceived effect of music on mental health | str | C / A crossover | 1.09% | Improve / No effect / Worsen |
| 32 | `Permissions` | Consent acknowledgement | str | E — Metadata | 0% | 100% "I understand." — Exclude |


In [ ]:
# Verify the Frequency column ordinal values
freq_cols = [c for c in df_mxmh.columns if c.startswith('Frequency')]
print("All unique values across all 16 Frequency columns:")
all_freq_vals = set()
for col in freq_cols:
    for val in df_mxmh[col].dropna().unique():
        all_freq_vals.add(val)
print(sorted(all_freq_vals))

print()
print("This confirms: all Frequency columns share exactly 4 ordinal categories.")


In [ ]:
# Inspect Music effects in detail
print("Music effects value counts:")
print(df_mxmh['Music effects'].value_counts(dropna=False))
print()
print("This column captures perceived direction of music on mental health.")
print("It straddles music behaviour (listening) and mental health (perceived effect).")


## Column Interpretations — music_brain_wellbeing.csv

| # | Column | Meaning | Type | Category | Missing % | Notes |
|---|---|---|---|---|---|---|
| 0 | `participant_id` | Unique participant identifier | str | E — Metadata / identifier | 0% | P001 appears twice — 1 duplicate row |
| 1 | `age` | Participant age in years | int64 | B — Demographic | 0% | Range 19–36 |
| 2 | `gender` | Gender identity | str | B — Demographic | 0% | Female / Male / Non-binary |
| 3 | `primary_genre` | Favourite music genre | str | A — Music behaviour | 0% | 5 genres: Ambient, Classical, Pop, Rock, Jazz |
| 4 | `daily_listening_hours` | Hours of music listening per day | float64 | A — Music behaviour | 0% | Range 1.2–5.0 |
| 5 | `skip_rate` | Fraction of songs skipped (0.0–1.0) | float64 | A — Music behaviour | 0% | Proxy for engagement/disengagement |
| 6 | `eeg_alpha_power` | EEG alpha wave power measurement | float64 | D — Physiological | 3.85% | 1 missing; higher = more relaxed state |
| 7 | `heart_rate_bpm` | Average heart rate during listening | float64 | D — Physiological | 3.85% | 1 missing; higher = more aroused state |
| 8 | `anxiety_score` | Self-reported anxiety score 0–10 | float64 | C — Mental health | 0% | Simulated self-report |
| 9 | `wellbeing_score` | Self-reported wellbeing score 0–100 | float64 | C — Mental health | 0% | **Note: different scale from MXMH** |


---
# Phase 4 — Data Quality Audit

We systematically check for quality issues: missing values, duplicates, impossible values,
suspicious outliers, inconsistent labels, and whitespace problems.

We do NOT automatically remove anything. We only identify and document.


In [ ]:
# 4.1 DUPLICATE ROWS
print("=== Duplicate Row Check ===")
print()
print("mxmh_survey_results.csv — duplicate rows:", df_mxmh.duplicated().sum())
print()
print("music_brain_wellbeing.csv — duplicate rows:", df_mbw.duplicated().sum())

# Show the duplicate row in df_mbw
if df_mbw.duplicated().sum() > 0:
    print()
    print("Duplicate row in music_brain_wellbeing.csv:")
    print(df_mbw[df_mbw.duplicated(keep=False)].to_string())
    print()
    print("Observation: participant_id P001 appears twice with identical values.")
    print("This is a data entry error — the same row was recorded twice.")


In [ ]:
# 4.2 BPM OUTLIERS — The most critical quality issue in mxmh
print("=== BPM Outlier Analysis ===")
print()
print("BPM descriptive statistics:")
print(df_mxmh['BPM'].describe().round(2))
print()

# Identify extreme BPM values
extreme_bpm = df_mxmh[df_mxmh['BPM'] > 250][['Age', 'Fav genre', 'BPM']]
print(f"Rows with BPM > 250: {len(extreme_bpm)}")
print(extreme_bpm.to_string())
print()
print("Observation: BPM = 999,999,999 is clearly a data entry error, not a real tempo.")
print("Typical music BPM ranges from ~40 (very slow) to ~200 (very fast).")
print("BPM = 624 is also highly implausible for a human's preferred listening tempo.")
print("These rows should be flagged and handled carefully — not blindly deleted yet.")

low_bpm = df_mxmh[df_mxmh['BPM'] < 40][['Age', 'Fav genre', 'BPM']]
print()
print(f"Rows with BPM < 40: {len(low_bpm)}")
print(low_bpm.to_string())


In [ ]:
# 4.3 AGE RANGE AUDIT
print("=== Age Range Audit ===")
print()
print("Age descriptive statistics:")
print(df_mxmh['Age'].describe().round(2))
print()
print("Min age:", df_mxmh['Age'].min(), " — age 10 is young but conceivable for an online survey")
print("Max age:", df_mxmh['Age'].max(), " — age 89 is plausible")
print("No ages below 0 or above 100 found.")
print()
print("Distribution of ages 10–14:")
young = df_mxmh[df_mxmh['Age'] < 15]
print(young[['Age', 'Fav genre', 'Anxiety']].to_string())


In [ ]:
# 4.4 HOURS PER DAY AUDIT
print("=== Hours per Day Audit ===")
print()
print("Hours per day descriptive statistics:")
print(df_mxmh['Hours per day'].describe().round(2))
print()

extreme_hours = df_mxmh[df_mxmh['Hours per day'] >= 20][['Age', 'Hours per day']]
print(f"Rows with Hours per day >= 20: {len(extreme_hours)}")
if len(extreme_hours) > 0:
    print(extreme_hours.to_string())
    print()
    print("Observation: 24 hours/day is technically possible but represents a boundary case.")
    print("It may be a maximum-click entry error, or the respondent simply answered '24'.")


In [ ]:
# 4.5 MENTAL HEALTH SCORE RANGE AUDIT
print("=== Mental Health Score Range Audit ===")
print()
for col in ['Anxiety', 'Depression', 'Insomnia', 'OCD']:
    min_val = df_mxmh[col].min()
    max_val = df_mxmh[col].max()
    out_of_range = df_mxmh[(df_mxmh[col] < 0) | (df_mxmh[col] > 10)][col].count()
    print(f"{col}: min={min_val}, max={max_val}, out-of-range (not 0-10): {out_of_range}")
print()
print("All four mental health columns are within the expected 0-10 range.")


In [ ]:
# 4.6 CATEGORICAL CONSISTENCY CHECK
print("=== Categorical Consistency Audit ===")
print()

cat_cols = ['Primary streaming service', 'While working', 'Instrumentalist',
            'Composer', 'Fav genre', 'Exploratory', 'Foreign languages',
            'Music effects']

for col in cat_cols:
    vals = df_mxmh[col].dropna().unique()
    # Check for whitespace padding or case inconsistencies
    stripped = [str(v).strip() for v in vals]
    lower    = [str(v).lower() for v in vals]
    has_ws   = any(str(v) != str(v).strip() for v in vals)
    has_case = len(set(lower)) < len(set([str(v) for v in vals]))
    print(f"{col}: {len(vals)} unique | whitespace={has_ws} | case_issue={has_case}")
    print(f"  Values: {sorted([str(v) for v in vals])}")
    print()


In [ ]:
# 4.7 PERMISSIONS COLUMN
print("=== Permissions Column ===")
print()
print("Unique values in Permissions column:")
print(df_mxmh['Permissions'].value_counts(dropna=False))
print()
print("This column contains 100% 'I understand.' responses.")
print("It is a consent acknowledgement checkbox — not a data variable.")
print("It should be excluded from all analysis.")


---
# Phase 5 — Understand the Target / Outcome Variables

This is critical. We must not confuse self-reported survey scores with clinical diagnoses.

**Important distinction**:
- **Clinical diagnosis**: Made by a qualified mental health professional using standardised diagnostic criteria (e.g. DSM-5). This dataset does NOT contain clinical diagnoses.
- **Self-reported survey score**: A participant rates their own symptom severity on a numerical scale. This is what we have.

The four mental health variables (`Anxiety`, `Depression`, `Insomnia`, `OCD`) are **self-reported symptom severity scores on a 0–10 scale**. They should always be described as such.


In [ ]:
# Inspect the four candidate target variables
print("=== Candidate Target Variables ===")
print()

target_cols = ['Anxiety', 'Depression', 'Insomnia', 'OCD']

for col in target_cols:
    print(f"--- {col} ---")
    print(f"  Type   : Self-reported symptom severity score (0 = none, 10 = maximum)")
    print(f"  Min    : {df_mxmh[col].min()}")
    print(f"  Max    : {df_mxmh[col].max()}")
    print(f"  Mean   : {df_mxmh[col].mean():.2f}")
    print(f"  Median : {df_mxmh[col].median():.2f}")
    print(f"  Missing: {df_mxmh[col].isnull().sum()} (0%)")
    print()

print("Also present: 'Music effects' — perceived effect of music on mental health")
print(df_mxmh['Music effects'].value_counts(dropna=False))


## Candidate Target Variable Roles

| Variable | Nature | Possible Role | Notes |
|---|---|---|---|
| `Anxiety` | Self-reported 0–10 | **Primary target candidate** | No missing values |
| `Depression` | Self-reported 0–10 | **Primary target candidate** | No missing values |
| `Insomnia` | Self-reported 0–10 | **Secondary target candidate** | No missing values |
| `OCD` | Self-reported 0–10 | **Secondary target candidate** | No missing values |
| `Music effects` | Categorical 3-class | **Classification target candidate** | 8 missing values |

We do NOT choose the final target here. That decision belongs to the next stage.
If predicting `Anxiety`, then `Depression`, `Insomnia`, and `OCD` become potential features or control variables,
but this requires careful reasoning about confounding — not handled at this stage.


---
# Phase 6 — Music Variables

We identify and conceptually organise all variables related to music behaviour.


In [ ]:
# Identify all music-related variables
print("=== Music-Related Variables ===")
print()

music_vars = {
    "Music Behaviour": ["Hours per day", "While working", "Instrumentalist", "Composer",
                        "Exploratory", "Foreign languages", "Primary streaming service"],
    "Music Preference": ["Fav genre", "BPM"],
    "Listening Frequency (per genre)": [c for c in df_mxmh.columns if c.startswith("Frequency")],
    "Self-Reported Response": ["Music effects"]
}

for group, cols in music_vars.items():
    print(f"--- {group} ---")
    for col in cols:
        print(f"  {col}")
    print()


## Conceptual Variable Organization

This is a conceptual grouping — NOT a causal diagram.

```
Music Behaviour
  (Hours per day, While working, Instrumentalist, Composer, Exploratory, Foreign languages)
         |
         v
Music Preference
  (Fav genre, BPM)
         |
         v
Listening Context
  (Primary streaming service)
         |
         v
Listening Frequency per Genre
  (Frequency [Classical], Frequency [Rock], ... x 16 genres)
         |
         v
Self-Reported Response to Music
  (Music effects: Improve / No effect / Worsen)
         |
         v
Wellbeing Indicators (Self-Reported)
  (Anxiety, Depression, Insomnia, OCD)
```

We are investigating whether variables higher in this hierarchy
carry useful statistical information about variables at the bottom.
We are NOT claiming any causal direction.


---
# Phase 7 — Basic Exploratory Data Analysis

Every visualisation answers a specific question.
We do not produce plots without a stated purpose.


In [ ]:
# Configure plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style="whitegrid")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: What is the age distribution of participants?
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_mxmh['Age'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")

axes[1].boxplot(df_mxmh['Age'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title("Age Boxplot")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.savefig("docs/figures/eda_age_distribution.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Most participants are 15-30 years old.")
print("The distribution is right-skewed — a long tail toward older ages.")
print("Median age is ~21. This is a young-adult-heavy sample.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: How many hours per day do participants listen to music?
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_mxmh['Hours per day'], bins=25, color='teal', edgecolor='white')
axes[0].set_title("Hours per Day Listening")
axes[0].set_xlabel("Hours per day")
axes[0].set_ylabel("Count")

axes[1].boxplot(df_mxmh['Hours per day'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='teal', alpha=0.6))
axes[1].set_title("Hours per Day Boxplot")
axes[1].set_xlabel("Hours per day")

plt.tight_layout()
plt.savefig("docs/figures/eda_hours_distribution.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Most participants listen 1-5 hours per day.")
print("The distribution is right-skewed with a few extreme values near 24.")
print("Median is ~3 hours per day.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: What is the distribution of self-reported mental health scores?
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
mental_cols = ['Anxiety', 'Depression', 'Insomnia', 'OCD']
colours = ['#e74c3c', '#8e44ad', '#2980b9', '#27ae60']

for ax, col, colour in zip(axes.flatten(), mental_cols, colours):
    ax.hist(df_mxmh[col], bins=11, range=(-0.5, 10.5), color=colour, edgecolor='white', alpha=0.85)
    ax.set_title(f"{col} Score Distribution")
    ax.set_xlabel(f"{col} (0 = none, 10 = severe)")
    ax.set_ylabel("Count")
    ax.set_xticks(range(11))

plt.suptitle("Self-Reported Mental Health Score Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("docs/figures/eda_mental_health_distributions.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Anxiety and Depression show broad distributions.")
print("Insomnia is more moderate. OCD is heavily right-skewed (most report low OCD).")
print("These are self-reported symptoms — we cannot treat them as clinical diagnoses.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: What is the favourite genre breakdown?
# ------------------------------------------------------------------
genre_counts = df_mxmh['Fav genre'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
genre_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title("Favourite Genre Distribution")
ax.set_xlabel("Genre")
ax.set_ylabel("Number of Participants")
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("docs/figures/eda_genre_distribution.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Rock is the most popular favourite genre (188 participants).")
print("Latin and Gospel are the least represented (3 and 6 participants).")
print("This imbalance will matter in modelling — rare genres may lack statistical power.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: What streaming service do participants use most?
# ------------------------------------------------------------------
service_counts = df_mxmh['Primary streaming service'].value_counts(dropna=False)

fig, ax = plt.subplots(figsize=(10, 5))
service_counts.plot(kind='bar', ax=ax, color='#2ecc71', edgecolor='white')
ax.set_title("Primary Streaming Service")
ax.set_xlabel("Service")
ax.set_ylabel("Count")
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig("docs/figures/eda_streaming_service.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Spotify dominates (458 out of 736).")
print("71 participants do not use a streaming service at all.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: How correlated are the four mental health scores?
# ------------------------------------------------------------------
mental_cols = ['Anxiety', 'Depression', 'Insomnia', 'OCD']
corr_matrix = df_mxmh[mental_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Mental Health Scores")

plt.tight_layout()
plt.savefig("docs/figures/eda_mental_health_corr.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Anxiety and Depression are moderately positively correlated (~0.5).")
print("This makes intuitive sense — respondents who report high anxiety often also report")
print("higher depression. OCD shows lower correlation with the others.")
print("We cannot conclude these correlations imply any causal relationship.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: Is listening time associated with music effects perception?
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
df_mxmh_clean = df_mxmh.dropna(subset=['Music effects'])
df_mxmh_clean.groupby('Music effects')['Hours per day'].mean().sort_values().plot(
    kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='white'
)
ax.set_title("Mean Hours per Day by Music Effects Perception")
ax.set_xlabel("Music Effects")
ax.set_ylabel("Mean Hours per Day")
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig("docs/figures/eda_hours_by_music_effects.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Participants who perceive music as worsening their mental state")
print("show a slightly different listening pattern than those who perceive improvement.")
print("These are group averages — we cannot conclude causality from this alone.")


---
# Phase 8 — Initial Relationship Analysis

We explore whether music-related variables show any patterns with wellbeing variables.
Language used: "associated with", "shows a pattern", "correlated with" — never "causes".


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: Is hours per day listening associated with Anxiety levels?
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(df_mxmh['Hours per day'], df_mxmh['Anxiety'],
           alpha=0.3, color='steelblue', edgecolors='none')
ax.set_title("Hours per Day vs Self-Reported Anxiety")
ax.set_xlabel("Hours per Day")
ax.set_ylabel("Anxiety Score (0-10)")

# Add a trend line using numpy polyfit
x = df_mxmh['Hours per day'].dropna()
y = df_mxmh['Anxiety'][x.index]
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.7, label=f'Trend (slope={z[0]:.3f})')
ax.legend()

plt.tight_layout()
plt.savefig("docs/figures/eda_hours_vs_anxiety.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Scatter is very wide — hours per day alone is a weak predictor.")
print("The trend line shows a very slight positive slope (more listening slightly associated")
print("with higher anxiety), but the relationship is weak. Confounders likely exist.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: Do anxiety scores differ across favourite genres?
# ------------------------------------------------------------------
genre_anxiety = (df_mxmh.groupby('Fav genre')['Anxiety']
                 .agg(['mean', 'count'])
                 .sort_values('mean', ascending=False))

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(genre_anxiety.index, genre_anxiety['mean'], color='salmon', edgecolor='white')
ax.set_title("Mean Self-Reported Anxiety Score by Favourite Genre")
ax.set_xlabel("Favourite Genre")
ax.set_ylabel("Mean Anxiety Score (0-10)")
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("docs/figures/eda_anxiety_by_genre.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Some genre groups show different mean anxiety scores.")
print("Metal and Rock listeners report slightly higher average anxiety than Classical.")
print("BUT: sample sizes differ widely across genres. Rare genres (Latin, Gospel, Lofi)")
print("have very few respondents, so their group means are unreliable estimates.")
print("We should not over-interpret genre differences without accounting for group size.")


In [ ]:
# ------------------------------------------------------------------
# Question we are asking: What is the correlation structure between
# music behaviour variables and mental health scores?
# ------------------------------------------------------------------
# Convert ordinal frequency strings to numbers for correlation computation
freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Very frequently': 3}
freq_cols = [c for c in df_mxmh.columns if c.startswith('Frequency')]

df_numeric = df_mxmh.copy()
for col in freq_cols:
    df_numeric[col] = df_numeric[col].map(freq_map)

# Select variables for correlation
numeric_for_corr = ['Hours per day', 'BPM'] + freq_cols[:8] + ['Anxiety', 'Depression', 'Insomnia', 'OCD']

# Filter to valid BPM range for correlation
df_corr = df_numeric[df_numeric['BPM'] < 300][numeric_for_corr]
corr = df_corr.corr()

# Show correlation with mental health outcomes only
mental_health_cols = ['Anxiety', 'Depression', 'Insomnia', 'OCD']
corr_with_mh = corr[mental_health_cols].drop(mental_health_cols)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_with_mh, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Correlation: Music Variables vs Mental Health Scores")
plt.tight_layout()
plt.savefig("docs/figures/eda_music_mh_correlation.png", dpi=120, bbox_inches='tight')
plt.show()
print("What we observe: Most correlations between music frequency variables and")
print("mental health scores are small (|r| < 0.20).")
print("This does not mean music variables are useless — it means simple linear")
print("correlations may not capture the full relationship structure.")


---
# Phase 9 — Dataset Comparison

Before any thought of merging, we must first compare the two datasets carefully.


In [ ]:
print("=== Dataset Comparison ===")
print()
comparison = {
    "Attribute"              : ["Source",         "Rows",   "Columns", "Participants",  "BPM column",        "Mental health variables",    "Scale: wellbeing",   "Physiological data", "Duplicates"],
    "mxmh_survey_results"   : ["Public survey",   "736",    "33",      "736 unique-ish","Self-reported",     "Anxiety/Depression/Insomnia/OCD (0-10)", "Unified 0-10",  "None",               "0"],
    "music_brain_wellbeing" : ["Synthetic sample","26 (25 unique)", "10", "25 unique",  "None",              "anxiety_score (0-10), wellbeing_score (0-100)", "Different (0-100)", "EEG, heart rate", "1"]
}

comp_df = pd.DataFrame(comparison).set_index("Attribute")
print(comp_df.to_string())
print()
print("=" * 70)
print("CONCLUSION: These two datasets CANNOT be merged.")
print("=" * 70)
print()
print("Reasons:")
print("1. Different populations: MXMH has 736 real survey respondents;")
print("   music_brain_wellbeing has 25 synthetic/simulated participants.")
print("2. Different variables: MXMH has no EEG or heart rate data;")
print("   music_brain_wellbeing has no genre frequency or streaming service data.")
print("3. Different scales: wellbeing_score uses 0-100; MXMH uses 0-10.")
print("4. Different sources: one is a public dataset; the other was created as a")
print("   practice sample and does not represent real data.")
print()
print("The primary working dataset is: mxmh_survey_results.csv")
print("music_brain_wellbeing.csv will be used for practice purposes only.")


---
# Notebook Complete

**Summary of key findings:**
1. `mxmh_survey_results.csv` — 736 rows × 33 columns. Primary working dataset.
2. `music_brain_wellbeing.csv` — 26 rows (25 unique) × 10 columns. Synthetic practice sample. Cannot be merged with MXMH.
3. **`BPM` column is seriously problematic** — contains 107 missing values (14.5%) and extreme outliers (999,999,999). Requires careful handling.
4. **Four mental health variables** (`Anxiety`, `Depression`, `Insomnia`, `OCD`) are self-reported 0–10 scores — NOT clinical diagnoses.
5. **`Permissions` column** contains 100% identical values — exclude from analysis.
6. **No duplicate rows** in MXMH; **1 duplicate row** (P001) in MBW.
7. **All 16 Frequency columns** share exactly 4 ordinal categories: Never / Rarely / Sometimes / Very frequently.
8. **No whitespace or capitalisation inconsistencies** detected in categorical columns.

**Next stage**: Data cleaning decisions — handling BPM outliers, encoding ordinal Frequency columns, treating missing values, and preparing a clean working dataset.
